# Validación robusta antes del test ciego

Confirma que la selección provisional de 44 variables es estable. Solo usa 2019–2021 para entrenar y 2022 para validar. **No cargar ni usar 2023.**

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.modeling.data import TARGET_COLUMN, TRAIN_YEARS, VALIDATION_YEARS, load_dataset_contract, sample_years_for_training
from src.modeling.evaluation import evaluate_binary_predictions
from src.modeling.experiments import fit_lightgbm_experiment
from src.modeling.features import resolve_feature_set

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
contract = load_dataset_contract(ROOT / 'data' / 'processed' / 'tabular' / 'egif')
OUTPUT_DIR = ROOT / 'outputs' / 'modeling'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert VALIDATION_YEARS == (2022,)
print('Train:', TRAIN_YEARS, '| Validación:', VALIDATION_YEARS, '| Test bloqueado: 2023')

## 1. Repetición con distintos negativos

Cada residuo toma todos los positivos y un 4 % distinto de negativos. Una diferencia pequeña y estable entre repeticiones da confianza; una diferencia grande indica que aún no debemos congelar variables.

In [ ]:
MODULUS = 25
REMAINDERS = (0, 7, 13, 19)
FEATURE_SETS = ('completo', 'temporal_compacto')
SEED = 42
rows = []

for remainder in REMAINDERS:
    train = sample_years_for_training(contract, TRAIN_YEARS, contract.predictors, MODULUS, remainder)
    validation = sample_years_for_training(contract, VALIDATION_YEARS, contract.predictors, MODULUS, remainder)
    for feature_set in FEATURE_SETS:
        predictors = resolve_feature_set(contract.predictors, feature_set)
        model, metrics = fit_lightgbm_experiment(train, validation, predictors, seed=SEED)
        rows.append({'remainder': remainder, 'feature_set': feature_set, 'n_predictors': len(predictors), **metrics})

stability = pd.DataFrame(rows)
summary = stability.groupby('feature_set')[['roc_auc', 'pr_auc', 'recall_at_top_fraction']].agg(['mean', 'std'])
display(stability.sort_values(['remainder', 'feature_set']))
display(summary)
assert set(stability['feature_set']) == set(FEATURE_SETS)

## 2. Estabilidad mensual del conjunto ganador

Se entrena una vez usando el primer residuo y se calculan métricas por mes sobre la validación 2022 muestreada. Estos valores comparan meses; no son tasas poblacionales finales.

In [ ]:
winner = summary[('pr_auc', 'mean')].idxmax()
predictors = resolve_feature_set(contract.predictors, winner)
train = sample_years_for_training(contract, TRAIN_YEARS, contract.predictors, MODULUS, REMAINDERS[0])
validation = sample_years_for_training(contract, VALIDATION_YEARS, contract.predictors, MODULUS, REMAINDERS[0])
model, _ = fit_lightgbm_experiment(train, validation, predictors, seed=SEED)
validation = validation.copy()
validation['probability'] = model.predict_proba(validation[predictors])[:, 1]
validation['month'] = pd.to_datetime(validation['fecha']).dt.month
monthly = []
for month, frame in validation.groupby('month'):
    if frame[TARGET_COLUMN].nunique() == 2:
        monthly.append({'month': int(month), 'rows': len(frame), 'positives': int(frame[TARGET_COLUMN].sum()), **evaluate_binary_predictions(frame[TARGET_COLUMN], frame['probability'])})
monthly = pd.DataFrame(monthly)
display(monthly)
monthly.plot(x='month', y=['pr_auc', 'recall_at_top_fraction'], marker='o', title=f'Estabilidad mensual: {winner}')

In [ ]:
payload = {'train_years': list(TRAIN_YEARS), 'validation_years': list(VALIDATION_YEARS), 'test_years_not_used': [2023], 'modulus': MODULUS, 'remainders': list(REMAINDERS), 'winner_by_mean_pr_auc': winner, 'stability': stability.to_dict(orient='records'), 'monthly': monthly.to_dict(orient='records')}
path = OUTPUT_DIR / 'robust_validation_2022.json'
path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(path)